# SQL Murder Mystery

## Can you find out whodunnit?

![A decorative illustration of a detective looking at an evidence board.](https://mystery.knightlab.com/174092-clue-illustration.png)

There's been a Murder in SQL City! The SQL Murder Mystery is designed to be both a self-directed lesson to learn SQL concepts and commands and a fun game for experienced SQL users to solve an intriguing crime.

## SQL sleuths start here

A crime has taken place and the detective needs your help. The detective gave you the crime scene report, but you somehow lost it. You vaguely remember that the crime was a **​murder​**that occurred sometime on ​**Jan.15, 2018​** and that it took place in ​**SQL City​**. Start by retrieving the corresponding crime scene report from the police department’s database.

### Exploring the Database Structure

Experienced SQL users can often use database queries to infer the structure of a database. But each database system has different ways of managing this information. The SQL Murder Mystery is built using SQLite. Use this SQL command to find the tables in the Murder Mystery database.

Run this query to find the names of the tables in this database.

This command is specific to SQLite. For other databases, you'll have to learn their specific syntax.


In [1]:
# Install required packages
%pip install jupysql sqlalchemy pandas --quiet

# Load SQL magic
%load_ext sql

# Connect to the database
%sql sqlite:///sql-murder-mystery.db

Note: you may need to restart the kernel to use updated packages.


Connecting to 'sqlite:///sql-murder-mystery.db'

In [2]:
%%sql
SELECT name
FROM sqlite_master
where type = 'table'

Running query in 'sqlite:///sql-murder-mystery.db'

name
crime_scene_report
drivers_license
facebook_event_checkin
interview
get_fit_now_member
get_fit_now_check_in
solution
income
person



Besides knowing the table names, you need to know how each table is structured. The way this works is also dependent upon which database technology you use. Here's how you do it with SQLite.

Run this query to find the structure of the `crime_scene_report` table

Change the value of 'name' to see the structure of the other tables you learned about with the previous query.


In [6]:
%%sql
SELECT sql
FROM sqlite_master
where name = 'crime_scene_report'

Running query in 'sqlite:///sql-murder-mystery.db'

sql
"CREATE TABLE crime_scene_report ( date integer, type text, description text, city text )"


In [7]:
%%sql
-- This finds the murder case on Jan 15, 2018 in SQL City
SELECT *
FROM crime_scene_report
WHERE date == 20180115 AND city == 'SQL City' AND type == 'murder';

Running query in 'sqlite:///sql-murder-mystery.db'

date,type,description,city
20180115,murder,"Security footage shows that there were 2 witnesses. The first witness lives at the last house on ""Northwestern Dr"". The second witness, named Annabel, lives somewhere on ""Franklin Ave"".",SQL City


In [12]:
%%sql
-- This uses first witness id from person and shows her info from interview
SELECT *
From interview
WHERE person_id = (
    SELECT id
    FROM person
    WHERE address_street_name = 'Northwestern Dr'
    ORDER BY address_number DESC 
    LIMIT 1 -- The last house at 'Northwestern Dr'
);


Running query in 'sqlite:///sql-murder-mystery.db'

person_id,transcript
14887,"I heard a gunshot and then saw a man run out. He had a ""Get Fit Now Gym"" bag. The membership number on the bag started with ""48Z"". Only gold members have those bags. The man got into a car with a plate that included ""H42W""."


In [13]:
%%sql
-- This uses second witness "Annabel" id from person and shows her info from interview
SELECT *
From interview
WHERE person_id IN (
    SELECT id
    FROM person
    WHERE name LIKE '%Annabel%' AND address_street_name == 'Franklin Ave');

Running query in 'sqlite:///sql-murder-mystery.db'

person_id,transcript
16371,"I saw the murder happen, and I recognized the killer from my gym when I was working out last week on January the 9th."


In [ ]:
%%sql
-- Find gold members at gym who started before the murder date and membership number starting with '48Z'
SELECT *
    FROM get_fit_now_member
    WHERE membership_start_date < 20180115
        AND membership_status = 'gold'
        AND id LIKE '48Z%'

Running query in 'sqlite:///sql-murder-mystery.db'

id,person_id,name,membership_start_date,membership_status
48Z7A,28819,Joe Germuska,20160305,gold
48Z55,67318,Jeremy Bowers,20160101,gold


In [ ]:
%%sql
-- Check if suspectswere at gym on murder date
SELECT *
FROM get_fit_now_check_in
WHERE membership_id IN (
    SELECT id
    FROM get_fit_now_member
    WHERE membership_start_date < 20180115
        AND membership_status = 'gold'
        AND id LIKE '48Z%');

Running query in 'sqlite:///sql-murder-mystery.db'

membership_id,check_in_date,check_in_time,check_out_time
48Z7A,20180109,1600,1730
48Z55,20180109,1530,1700


In [ ]:
%%sql
-- Find suspects personal info
SELECT *
FROM person
WHERE name = "Joe Germuska" OR  name = "Jeremy Bowers";

Running query in 'sqlite:///sql-murder-mystery.db'

id,name,license_id,address_number,address_street_name,ssn
28819,Joe Germuska,173289,111,Fisk Rd,138909730
67318,Jeremy Bowers,423327,530,"Washington Pl, Apt 3A",871539279


In [ ]:
%%sql
-- Check to see if the suspects have alibis
SELECT *
FROM facebook_event_checkin
WHERE person_id = 28819 OR person_id = 67318;

Running query in 'sqlite:///sql-murder-mystery.db'

person_id,event_id,event_name,date
67318,4719,The Funky Grooves Tour,20180115
67318,1143,SQL Symphony Concert,20171206


In [ ]:
%%sql
-- Check whether suspect "Jeremy Bowers" car plate with murderers car plate (includes "H42W")
SELECT plate_number
FROM drivers_license
WHERE id in (
    SELECT license_id

    FROM person
    WHERE name = "Jeremy Bowers"
);

Running query in 'sqlite:///sql-murder-mystery.db'

plate_number
0H42W2


In [21]:
%%sql
-- Find "Jeremy Bowers" interview
SELECT *
FROM interview
WHERE person_id = 67318;

Running query in 'sqlite:///sql-murder-mystery.db'

person_id,transcript
67318,"I was hired by a woman with a lot of money. I don't know her name but I know she's around 5'5"" (65"") or 5'7"" (67""). She has red hair and she drives a Tesla Model S. I know that she attended the SQL Symphony Concert 3 times in December 2017."


In [26]:
%%sql
-- Find murder mastermind info using murderer interview
SELECT *
FROM person
WHERE license_id IN (
    SELECT id
    FROM drivers_license
    WHERE 65 <= height <= 67
        AND gender = "female"
        AND hair_color = 'red'
        AND car_make LIKE 'Tesla'
        AND car_model LIKE 'Model S'
);

Running query in 'sqlite:///sql-murder-mystery.db'

id,name,license_id,address_number,address_street_name,ssn
78881,Red Korb,918773,107,Camerata Dr,961388910
90700,Regina George,291182,332,Maple Ave,337169072
99716,Miranda Priestly,202298,1883,Golden Ave,987756388


In [28]:
%%sql
-- Find which one above went to SQL Symphony Concert in December 2017
SELECT *
FROM facebook_event_checkin
WHERE person_id in (99716, 78881, 90700);


Running query in 'sqlite:///sql-murder-mystery.db'

person_id,event_id,event_name,date
99716,1143,SQL Symphony Concert,20171206
99716,1143,SQL Symphony Concert,20171212
99716,1143,SQL Symphony Concert,20171229


In [32]:
%%sql
-- Name of the mastermind behind this murder
SELECT name
FROM person
WHERE id = 99716;

Running query in 'sqlite:///sql-murder-mystery.db'

name
Miranda Priestly



### The rest is up to you!

If you're really comfortable with SQL, you can probably get it from here. To help, here is the schema diagram:

![schema diagram](schema.png)

Use your knowledge of the database schema and SQL commands to find out who committed the murder.
### Check your solution

Did you find the killer? When you think you know the answer, submit your suspect using the following code and find out if you're right.


In [33]:
%%sql
INSERT INTO solution VALUES (1, 'Miranda Priestly');
SELECT value FROM solution;

Running query in 'sqlite:///sql-murder-mystery.db'

1 rows affected.

value
"Congrats, you found the brains behind the murder! Everyone in SQL City hails you as the greatest SQL detective of all time. Time to break out the champagne!"
